# Comparing self-supervised representations

Three self-supervised pretext tasks are compared on CIFAR-10, using models pretrained by the
[Berkeley CS294-158 course](https://sites.google.com/view/berkeley-cs294-158-sp20/home):

| Method | Pretext task | Representation |
|--------|--------------|----------------|
| **Context Encoder** (Pathak et al., 2016) | inpaint the masked center of a 128×128 image | 4000-d bottleneck |
| **Rotation prediction** (Gidaris et al., 2018) | predict the rotation (0/90/180/270°) of an image | NIN conv2 feature map |
| **SimCLR** (Chen et al., 2020) | match two augmented views of the same image (contrastive) | ResNet-50 features (2048-d) |

Each representation is **frozen** and evaluated with a **linear classifier** (top-1/top-5 and
per-class accuracy) and with nearest neighbours in latent space. Finally, the CIFAR-10
classifiers are evaluated on **STL-10** to measure robustness to a domain shift.

> Requires a GPU and ~6.5 GB of pretrained weights and data (downloaded below).

## Setup
On Colab the next cell clones the course helper package and downloads the pretrained models. Locally, run the notebook from the `cs294-158-ssl` folder with this repository on your `PYTHONPATH`.

In [ ]:
import os
import sys

REPO_DIR = "/content/self-supervised-learning"
COURSE_DIR = "/content/cs294-158-ssl"

if "google.colab" in sys.modules:
    !git clone -q https://github.com/<your-username>/self-supervised-learning.git {REPO_DIR}
    !git clone -q https://github.com/ckurtz/cs294-158-ssl {COURSE_DIR}
    !pip install -q {COURSE_DIR}
    %cd {COURSE_DIR}
    !wget -q https://camille-kurtz.com/teaching/data.zip && unzip -qq data.zip && rm data.zip
    !wget -q https://camille-kurtz.com/teaching/results.zip && unzip -qq results.zip && rm results.zip
    sys.path.append(REPO_DIR)

In [ ]:
import pandas as pd
import torch
from deepul_helper.demos import (display_nearest_neighbors, load_model_and_data,
                                 show_context_encoder_inpainting)

from ssl_eval import (CIFAR10_CLASSES, evaluate_frozen_model, plot_grouped_bars,
                      stl10_transfer_loader)

%matplotlib inline

device = "cuda"
TASKS = {"context_encoder": "Context Encoder", "rotation": "Rotation", "simclr": "SimCLR"}
OUTPUT_DIR = os.path.join(REPO_DIR if "google.colab" in sys.modules else ".", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Context Encoder

The network sees an image whose 64×64 center is masked and reconstructs it. The loss is the
reconstruction error on the masked region $\hat{M}$:

$$\mathcal{L}_{rec} = \left\Vert \hat{M} \odot (x - F((1 - \hat{M})\odot x)) \right\Vert^2_2$$

(The original paper adds an adversarial loss, not used here.) Left: masked input, right: inpainting.

In [ ]:
show_context_encoder_inpainting()

## 2. Rotation prediction

Each image is rotated by 0°, 90°, 180° and 270°, and the network classifies the rotation. To
succeed it must recognise object orientation, which requires semantic features.

## 3. SimCLR

Two augmentations $x_i, x_j$ of the same image form a positive pair; the other images of the batch
are negatives. With $\text{sim}$ the cosine similarity and $\tau$ a temperature:

$$\ell_{i,j} = -\log{\frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k=1}^{2N}\mathbb{1}_{[k\neq i]}\exp(\text{sim}(z_i, z_k)/\tau)}}$$

## 4. Linear evaluation on CIFAR-10
Train accuracy is measured with the training augmentations of each task.

In [ ]:
results, models = {}, {}
for task, name in TASKS.items():
    model, classifier, train_loader, test_loader = load_model_and_data(task)
    models[task] = (model, classifier)
    results[name] = {
        "train": evaluate_frozen_model(model, classifier, train_loader, device),
        "test": evaluate_frozen_model(model, classifier, test_loader, device),
    }
    print(f"{name:<16} test top-1 {results[name]['test']['top1']:.2f}% | "
          f"top-5 {results[name]['test']['top5']:.2f}%")

table = pd.DataFrame({
    name: {"train top-1": r["train"]["top1"], "test top-1": r["test"]["top1"],
           "train top-5": r["train"]["top5"], "test top-5": r["test"]["top5"]}
    for name, r in results.items()
}).T.round(2)
table.to_csv(os.path.join(OUTPUT_DIR, "cifar10_linear_eval.csv"))
table

In [ ]:
names = list(results)
plot_grouped_bars({"Top-1": [results[n]["test"]["top1"] for n in names],
                   "Top-5": [results[n]["test"]["top5"] for n in names]},
                  names, title="Linear evaluation on CIFAR-10 (test set)", ylim=(0, 105),
                  save_path=os.path.join(OUTPUT_DIR, "cifar10_topk.png"))

plot_grouped_bars({n: results[n]["test"]["per_class"] for n in names}, CIFAR10_CLASSES,
                  title="Per-class accuracy on CIFAR-10 (test set)", ylim=(0, 105),
                  save_path=os.path.join(OUTPUT_DIR, "cifar10_per_class.png"));

## 5. Nearest neighbours in latent space
For random test images, the 16 closest test images in representation space.

In [ ]:
for task, name in TASKS.items():
    print(f"===== {name} =====")
    _, _, _, test_loader = load_model_and_data(task)
    display_nearest_neighbors(task, models[task][0], test_loader, n_examples=2)

## 6. Transfer to STL-10 (domain shift)

The CIFAR-10 linear classifiers are applied, without retraining, to STL-10 test images (96×96,
higher resolution, different photos).

STL-10 and CIFAR-10 share 9 classes **in a different order** (e.g. STL-10 label 1 is *bird*,
CIFAR-10 label 1 is *automobile*), and STL-10's *monkey* has no CIFAR-10 counterpart. Labels are
therefore remapped to CIFAR-10 indices and *monkey* images are removed; images go through each
task's CIFAR-10 evaluation transform (same normalisation, no augmentation).

In [ ]:
transfer = {}
for task, name in TASKS.items():
    model, classifier = models[task]
    transfer[name] = evaluate_frozen_model(model, classifier, stl10_transfer_loader(task), device)
    print(f"{name:<16} STL-10 top-1 {transfer[name]['top1']:.2f}% | top-5 {transfer[name]['top5']:.2f}%")

shift = pd.DataFrame({n: {"CIFAR-10 top-1": results[n]["test"]["top1"],
                          "STL-10 top-1": transfer[n]["top1"],
                          "drop (pts)": results[n]["test"]["top1"] - transfer[n]["top1"]}
                      for n in names}).T.round(2)
shift.to_csv(os.path.join(OUTPUT_DIR, "stl10_transfer.csv"))
shift

In [ ]:
plot_grouped_bars({"CIFAR-10": [results[n]["test"]["top1"] for n in names],
                   "STL-10": [transfer[n]["top1"] for n in names]},
                  names, title="Top-1 accuracy under domain shift", ylim=(0, 105),
                  save_path=os.path.join(OUTPUT_DIR, "stl10_transfer.png"))

shared = [c for c in CIFAR10_CLASSES if c != "frog"]
idx = [CIFAR10_CLASSES.index(c) for c in shared]
plot_grouped_bars({n: transfer[n]["per_class"][idx] for n in names}, shared,
                  title="Per-class accuracy on STL-10", ylim=(0, 105),
                  save_path=os.path.join(OUTPUT_DIR, "stl10_per_class.png"));

## Observations

*Fill in after running the notebook.* Points worth discussing:

- the ranking of the three methods and the gap between contrastive learning and the older pretext tasks;
- which classes are hardest for each representation (cat / dog, bird / airplane...);
- how much each representation loses under the CIFAR-10 → STL-10 shift, and why
  (resolution change, the 32×32 downsampling used by Rotation and SimCLR, image statistics).